In [8]:
import pandas as pd
import xml.etree.ElementTree as ET
import json
from pathlib import Path

## 1. РЕЛЯЦИОННЫЙ ДАТАСЕТ

### crypto_market_cycles_2017_present.csv

**Что исследуем?** - Цены, доходность, волатильность, рыночные циклы

In [9]:
df_rel = pd.read_csv("crypto_market_cycles_2017_present.csv", nrows=5)

print(f"Формат: CSV (реляционный, табличный)")
print(f"Структура: {len(df_rel.columns)} столбцов")
print(f"Первая строка:\n{df_rel.iloc[0]}")
print(f"\nСтолбцы ({len(df_rel.columns)}):")
for i, col in enumerate(df_rel.columns, 1):
    print(f"  {i:2d}. {col}")

# Категориальные / числовые / временные
cat_cols = ['coin', 'fear_greed_label', 'fear_greed_zone', 'halving_epoch', 'golden_cross']
num_cols = ['open', 'high', 'low', 'close', 'volume', 'daily_return_pct', 'log_return',
            'rolling_7d_vol', 'rolling_30d_vol', 'ath_usd', 'drawdown_from_ath_pct',
            'sma_50d', 'sma_200d', 'fear_greed_value', 'btc_dominance_pct',
            'eth_dominance_pct', 'total_market_cap_usd', 'btc_market_cap_usd',
            'eth_market_cap_usd', 'days_since_halving', 'days_to_next_halving',
            'cycle_progress_pct', 'block_reward_btc', 'dxy_close', 'us10y_yield']

print(f"\nТипы характеристик:")
print(f"  • Временная: date (2017-11-09)")
print(f"  • Категориальные ({len(cat_cols)}): {cat_cols}")
print(f"  • Числовые ({len(num_cols)}): {num_cols[:8]}... и др.")
print(f"\nОбъект исследования: {df_rel['coin'].nunique()} монет, мульти-циклический период")

Формат: CSV (реляционный, табличный)
Структура: 32 столбцов
Первая строка:
date                               2017-11-09
coin                                      ADA
open                                  0.02516
high                                  0.03506
low                                  0.025006
close                                0.032053
volume                               18716200
daily_return_pct                          NaN
log_return                                NaN
rolling_7d_vol                            NaN
rolling_30d_vol                           NaN
ath_usd                              0.032053
drawdown_from_ath_pct                     0.0
sma_50d                              0.032053
sma_200d                             0.032053
golden_cross                                0
fear_greed_value                          NaN
fear_greed_label                          NaN
fear_greed_zone                           NaN
btc_dominance_pct                       36.24
eth_d

Первая группа — рыночные цены и объёмы.

Сюда относятся date, coin, open, high, low, close, volume.
Эти показатели позволяют исследовать поведение криптовалюты в течение торгового дня.

Вторая группа — показатели волатильности.
Особенно важны:
- daily_return_pct — дневная процентная доходность;
- log_return — логарифмическая доходность;
- rolling_7d_vol — скользящая волатильность за 7 дней;
- rolling_30d_vol — скользящая волатильность за 30 дней;
- drawdown_from_ath_pct — снижение цены относительно исторического максимума.

Третья группа — технические индикаторы.

Поля sma_50d, sma_200d, golden_cross, ath_usd позволяют искать особенности поведения рынка перед ростом, падением или сменой тренда.

Четвёртая группа — рыночный контекст и макроэкономика.

Поля btc_dominance_pct, eth_dominance_pct, total_market_cap_usd, dxy_close, us10y_yield характеризуют состояние рынка в целом и внешние экономические условия.

Пятая группа — рыночные циклы Bitcoin.

Поля halving_epoch, days_since_halving, days_to_next_halving, is_halving_day, cycle_progress_pct, block_reward_btc позволяют исследовать волатильность в разные периоды относительно халвингов.

## 2. ИЕРАРХИЧЕСКИЙ ДАТАСЕТ (XML)

### crypto-tradeing bots.xml

**Что исследуем?** - Торговые боты, алгоритмы, стратегии и поддерживаемые блокчейны

In [10]:
tree = ET.parse("crypto-tradeing bots.xml")
root = tree.getroot()

print(f"Формат: XML (иерархический)")
print(f"Корневой тег: <{root.tag}>")
print(f"Всего записей: {root.attrib.get('total_records', 'N/A')}")
print(f"\nРазделы (sections):")
for section in root.findall('section'):
    print(f"  • record_type={section.attrib.get('record_type'):20s} count={section.attrib.get('count')}")

# Пример одного record
sample = root.find('.//record[@id="profile_trojan"]')
print(f"\nПример записи (Trojan):")
for child in sample[:8]:
    val = (child.text or "")[:70]
    print(f"  {child.tag:15s}: {val}")

Формат: XML (иерархический)
Корневой тег: <dataset>
Всего записей: 220

Разделы (sections):
  • record_type=bot_profile          count=27
  • record_type=chain_profile        count=12
  • record_type=verified_fact        count=78
  • record_type=qa_pair              count=55
  • record_type=llm_consensus        count=44
  • record_type=real_test_data       count=4

Пример записи (Trojan):
  record_type    : bot_profile
  question       : What is Trojan, which chains does it support, and what tools does it o
  answer         : Category: DEX Sniper / Copy Trading | Chain focus: Solana | Chains (1)
  bot_category   : DEX Sniper / Copy Trading
  chain          : Solana
  section        : bot_profile
  model          : 
  provider       : 


### **Как связан с основным датасетом?**

XML помогает установить, какие типы стратегий существуют и в каких блокчейнах они применяются. Затем эти стратегии можно сопоставлять с рыночными режимами, обнаруженными в CSV и JSON.

Но здесь есть важное ограничение - по этому XML нельзя достоверно вычислить корреляцию между активностью конкретного бота и волатильностью криптовалюты, поскольку файл не содержит полноценного временного журнала сделок всех ботов.

Следовательно, наиболее обоснованная роль XML в твоей работе — качественная интерпретация торговых стратегий и построение классификации инструментов, а не прямая оценка их фактического влияния на цену.

## 3. JSON ДАТАСЕТ

### btcusdt_4h_fear_greed.json

**Что исследуем?** - Четырёхчасовые изменения BTC и индекс страха и жадности

In [11]:
with open("btcusdt_4h_fear_greed.json") as f:
    data = json.load(f)

# Если это список записей или одиночный объект
sample = data[0] if isinstance(data, list) else data
n_records = len(data) if isinstance(data, list) else 1

print(f"Формат: JSON")
print(f"Записей: {n_records}")
print(f"Пример структуры:")
for k, v in sample.items():
    print(f"  {k:30s}: {v}")

print(f"\nТипы характеристик:")
print(f"  • Временная: timestamp (4-часовые свечи)")
print(f"  • Числовые: open, close, high, low, Fear & Greed Index")
print(f"  • Категориальная: Fear & Greed Classification")
print(f"\nФакторы влияния: Fear & Greed Index — рыночный сентимент")

Формат: JSON
Записей: 14280
Пример структуры:
  timestamp                     : 2026-09-30 04:00:00
  open                          : 83640.0
  close                         : 83269.0
  high                          : 83685.0
  low                           : 83164.0
  Fear & Greed Index            : 71.0
  Fear & Greed Classification   : Greed

Типы характеристик:
  • Временная: timestamp (4-часовые свечи)
  • Числовые: open, close, high, low, Fear & Greed Index
  • Категориальная: Fear & Greed Classification

Факторы влияния: Fear & Greed Index — рыночный сентимент


### **Как связан с основным датасетом?**

Пересечение с первым датасетом здесь связь наиболее очевидна.

В CSV присутствуют date, coin, close, daily_return_pct, rolling_7d_vol, fear_greed_value.

В JSON присутствуют timestamp, close, Fear & Greed Index.

Данные можно объединить по дате и BTC, предварительно приведя четырёхчасовые наблюдения к дневному уровню.

Это позволит сопоставить долгосрочные показатели волатильности с краткосрочной динамикой и настроениями инвесторов.

## 4. НЕСТРУКТУРИРОВАННЫЙ ДАТАСЕТ

### crypto_news.csv

**Что исследуем?** - Новости, информационный фон и возможные рыночные реакции

In [12]:
df_news = pd.read_csv("crypto_news.csv")

print(f"Формат: CSV с текстовыми полями (неструктурированный текст)")
print(f"Записей: {len(df_news)}")
print(f"Столбцы: {list(df_news.columns)}")

print(f"\nИсточники (source):")
print(df_news['source'].value_counts().to_string())

print(f"\nВременной диапазон: {df_news['createdAt'].min()} → {df_news['createdAt'].max()}")

print(f"\nПример заголовка:")
print(f"  {df_news['title'].iloc[0][:100]}...")

print(f"\nТипы характеристик:")
print(f"  • Временная: createdAt")
print(f"  • Категориальные: source (coindesk, theguardian, decrypt, ...)")
print(f"  • Текстовые (неструктурированные): title, description")

Формат: CSV с текстовыми полями (неструктурированный текст)
Записей: 413
Столбцы: ['url', 'title', 'description', 'thumbnail', 'createdAt', 'source']

Источники (source):
source
cryptodaily        200
bsc                 40
cryptopotato        36
decrypt             35
cointelegraph       30
coindesk            25
theguardian         20
bitcoinmagazine     10
coinjournal          9
bitcoinist           8

Временной диапазон: Fri, 03 Jul 2026 16:55:53 GMT → Wed, 29 Jul 2026 13:57:00 GMT

Пример заголовка:
  Robot maker Unitree is going public. Hyperliquid traders see 4x upside from IPO price...

Типы характеристик:
  • Временная: createdAt
  • Категориальные: source (coindesk, theguardian, decrypt, ...)
  • Текстовые (неструктурированные): title, description


### **Как связан с основным датасетом?**

Основная задача — превратить текст в числовые признаки, пригодные для объединения с рыночными данными.

Например, с помощью NLP можно классифицировать новости по тональности:

Позитивная - рост интереса, инвестиции, принятие криптовалют

Нейтральная	- Информационные сообщения без явной оценки

Негативная - Взломы, ограничения, потери, кризисы

После классификации можно рассчитать среднюю тональность публикаций по каждой криптовалюте за день, количество негативных новостей, количество публикаций или долю сообщений о рисках. А затем сравнить эти показатели с фактической динамикой рынка.